# Masked Reconstruction Training Notebook

This notebook is self-contained so you can upload a single file to JupyterHub and run the masked reconstruction pretraining workflow without the local Python package layout.

Structure:

1. one code cell for each source file the workflow depends on
2. one configuration cell where you edit the run arguments
3. separate execution cells for data loading, model setup, training, checkpointing, and visualization

Required packages in the JupyterHub environment: `torch`, `torchvision`, `datasets`, `pillow`, `numpy`, `matplotlib`, `pandas`, `tqdm`.


In [ ]:
# Notebook-only setup imports
import csv
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import torch

RUN_ROOT = Path('.')
print('Working directory:', RUN_ROOT.resolve())


## File Cell: `dataloader.py`

In [ ]:
# data.py
"""
AI4Mars dataloading utilities.

I used the Hugging Face dataset `hassanjbara/AI4MARS`, which provides:

- `image`: original rover image (Navcam, PIL-like).
- `label_mask`: semantic segmentation mask with terrain classes encoded as:

    * 0 -> soil
    * 1 -> bedrock
    * 2 -> sand
    * 3 -> big rock
    * 255 -> null / no label

This module wraps the dataset into PyTorch `Dataset` and `DataLoader` objects,
adds basic preprocessing (resize, grayscale/RGB conversion, tensor conversion),
and provides optional scanning to remove a small number of corrupted samples.
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from typing import List

import numpy as np
from PIL import Image, UnidentifiedImageError

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

from datasets import load_dataset, load_from_disk  # add load_from_disk if you want on-disk caching

AI4MARS_CLASS_NAMES: List[str] = ["soil", "bedrock", "sand", "big_rock"]
AI4MARS_IGNORE_INDEX: int = 255


class AI4MarsHFDataset(Dataset):
    """
    PyTorch wrapper around a Hugging Face split of the AI4Mars dataset.

    This class:

    - Applies resizing and grayscale/RGB conversion to images.
    - Resizes segmentation masks with nearest-neighbor interpolation.
    - Maps unknown label values to an ignore index (`AI4MARS_IGNORE_INDEX`).
    - Optionally scans the split once to drop corrupted or undecodable samples
      and caches the list of valid indices for future runs.

    Parameters
    ----------
    hf_split :
        A single split of the Hugging Face dataset (e.g. `raw["train"]`).
    image_size : int, default=256
        Target spatial size (height and width) for images and masks.
    to_rgb : bool, default=False
        If ``True``, convert images to 3-channel RGB tensors.
        If ``False``, keep them as 1-channel grayscale tensors.
    scan_spurious : bool, default=False
        If ``True``, scan the split to find valid (decodable) samples, build
        ``valid_indices``, and save them to disk in ``cache_dir``. This is
        useful for the **first** run on a new dataset cache.
        If ``False``, the dataset attempts to load precomputed valid indices
        from disk and skips the expensive scan.
    cache_dir : str, default="./ai4mars_valid_indices"
        Directory where per-split valid index files are stored
        (e.g. ``valid_indices_train.npy``).
    split_name : str, default="train"
        Name of the split (``"train"``, ``"val"``, or ``"test"``) used to build
        the cache file name.

    Attributes
    ----------
    ds :
        The underlying Hugging Face dataset split.
    valid_indices : list[int]
        Indices of valid (non-corrupted) samples in ``ds``.
    image_transform :
        Composed torchvision transform applied to images.
    mask_resize :
        torchvision transform used to resize label masks.
    cache_path : str
        Path to the ``.npy`` file storing ``valid_indices`` for this split.
    """

    def __init__(
        self,
        hf_split,
        image_size: int = 256,
        to_rgb: bool = False,
        scan_spurious: bool = False,
        cache_dir: str = "./ai4mars_valid_indices",
        split_name: str = "train",
    ):
        super().__init__()
        self.ds = hf_split
        self.image_size = image_size
        self.to_rgb = to_rgb

        # --- transforms ---
        if to_rgb:
            self.image_transform = transforms.Compose(
                [
                    transforms.Resize(
                        (image_size, image_size),
                        interpolation=InterpolationMode.BILINEAR,
                    ),
                    transforms.Grayscale(num_output_channels=3),
                    transforms.ToTensor(),
                ]
            )
        else:
            self.image_transform = transforms.Compose(
                [
                    transforms.Resize(
                        (image_size, image_size),
                        interpolation=InterpolationMode.BILINEAR,
                    ),
                    transforms.Grayscale(num_output_channels=1),
                    transforms.ToTensor(),
                ]
            )

        self.mask_resize = transforms.Resize(
            (image_size, image_size),
            interpolation=InterpolationMode.NEAREST,
        )

        # --- caching for valid indices ---
        os.makedirs(cache_dir, exist_ok=True)
        self.cache_path = os.path.join(
            cache_dir, f"valid_indices_{split_name}_n{len(self.ds)}.npy"
        )
        legacy_cache_path = os.path.join(
            cache_dir, f"valid_indices_{split_name}.npy"
        )

        self.valid_indices: List[int] = []

        cache_loaded = False
        if not scan_spurious:
            for candidate_cache_path in (self.cache_path, legacy_cache_path):
                if not os.path.exists(candidate_cache_path):
                    continue

                candidate_indices = np.load(candidate_cache_path).astype(int).tolist()
                if not all(0 <= idx < len(self.ds) for idx in candidate_indices):
                    print(
                        "[AI4MarsHFDataset] Ignoring cached valid indices from "
                        f"{candidate_cache_path} because they do not match the current "
                        f"{split_name} split size ({len(self.ds)} samples)."
                    )
                    continue

                self.valid_indices = candidate_indices
                cache_loaded = True

                if candidate_cache_path != self.cache_path:
                    np.save(
                        self.cache_path,
                        np.array(self.valid_indices, dtype=np.int64),
                    )

                print(
                    f"[AI4MarsHFDataset] Loaded {len(self.valid_indices)} valid indices "
                    f"from cache: {candidate_cache_path}"
                )
                break

        if not cache_loaded:
            # Slow path: scan HF split and build valid_indices
            print("[AI4MarsHFDataset] Scanning for corrupted samples...")
            for i in range(len(self.ds)):
                try:
                    sample = self.ds[i]
                    img = sample.get("image", None)
                    mask = sample.get("label_mask", None)

                    if img is None or mask is None:
                        continue

                    # Force PIL to decode image / mask (may raise UnidentifiedImageError)
                    _ = img.size
                    _ = mask.size

                    self.valid_indices.append(i)
                except UnidentifiedImageError:
                    print(
                        f"[AI4MarsHFDataset] Skipping corrupted sample at index {i}"
                    )
                    continue

            print(
                f"[AI4MarsHFDataset] Kept {len(self.valid_indices)} / {len(self.ds)} samples"
            )

            # Persist valid indices so future runs can skip scanning
            np.save(self.cache_path, np.array(self.valid_indices, dtype=np.int64))
            print(
                f"[AI4MarsHFDataset] Saved valid indices to cache: {self.cache_path}"
            )

    def __len__(self) -> int:
        """Return the number of valid samples in this split."""
        return len(self.valid_indices)

    def __getitem__(self, idx: int):
        """
        Get a single (image, mask) pair.

        Parameters
        ----------
        idx : int
            Index in the filtered dataset (0 <= idx < len(self)).

        Returns
        -------
        img_t : torch.Tensor
            Image tensor of shape ``[C, H, W]``, where ``C`` is 1 (grayscale)
            or 3 (RGB) depending on ``to_rgb``.
        mask_t : torch.Tensor
            Integer segmentation mask of shape ``[H, W]`` with label values
            in ``{0, 1, 2, 3, AI4MARS_IGNORE_INDEX}``.
        """
        real_idx = self.valid_indices[idx]
        sample = self.ds[real_idx]

        img = sample["image"]
        mask = sample["label_mask"]

        # HF Image usually returns PIL.Image already, but keep numpy fallback
        if isinstance(img, np.ndarray):
            img = Image.fromarray(img.astype(np.uint8))
        if isinstance(mask, np.ndarray):
            mask = Image.fromarray(mask.astype(np.uint8))

        img_t = self.image_transform(img)  # [C,H,W]

        mask_resized = self.mask_resize(mask)
        mask_np = np.array(mask_resized, dtype=np.uint8)

        if mask_np.ndim == 3:
            mask_np = mask_np[..., 0]

        # Optional safety: clamp unknown labels to ignore_index
        valid_classes = [0, 1, 2, 3, AI4MARS_IGNORE_INDEX]
        mask_np = np.where(
            np.isin(mask_np, valid_classes), mask_np, AI4MARS_IGNORE_INDEX
        )

        mask_t = torch.from_numpy(mask_np.astype(np.int64))  # [H,W], long

        return img_t, mask_t


@dataclass
class DataLoaders:
    """
    Container for the three data splits used in training and evaluation.

    Attributes
    ----------
    - **train**: PyTorch ``DataLoader`` for the training split.
    - **val**: PyTorch ``DataLoader`` for the validation split.
    - **test**: PyTorch ``DataLoader`` for the test split.
    """

    train: DataLoader
    val: DataLoader
    test: DataLoader


def create_ai4mars_dataloaders(
    batch_size: int = 4,
    image_size: int = 256,
    num_workers: int = 4,
    val_fraction: float = 0.1,
    to_rgb: bool = False,
    seed: int = 42,
    cache_dir: str | None = None,
    max_train_samples: int | None = None,
    max_val_samples: int | None = None,
    max_test_samples: int | None = None,
    use_local_disk_copy: bool = False,
    local_disk_path: str = "./data/ai4mars_hf",
    scan_spurious: bool = False,
    valid_indices_cache_dir: str = "./ai4mars_valid_indices",
) -> DataLoaders:
    """
    Create train/validation/test dataloaders for the AI4Mars dataset.

    This function:

    - Downloads (or loads from disk) the Hugging Face dataset
      ``"hassanjbara/AI4MARS"``.
    - Splits it into train/val/test splits (using `val_fraction` and a fixed seed).
    - Optionally subsamples each split for faster experiments.
    - Wraps each split in an :class:`AI4MarsHFDataset`, which can scan for and
      cache valid (non-corrupted) samples.
    - Returns PyTorch dataloaders for each split.

    Parameters
    ----------
    batch_size : int, default=4
        Batch size used for all three dataloaders.
    image_size : int, default=256
        Spatial resolution to which images and masks are resized (square).
    num_workers : int, default=4
        Number of worker processes for data loading.
    val_fraction : float, default=0.1
        Fraction of the (non-test) data to reserve for validation.
    to_rgb : bool, default=False
        If ``True``, convert grayscale Navcam images to 3-channel RGB.
        If ``False``, keep them as 1-channel grayscale.
    seed : int, default=42
        Random seed used for splitting into train/val/test.
    cache_dir : str or None, default=None
        Directory used by Hugging Face to cache the raw dataset.
        If ``None``, the default HF cache location is used.
    max_train_samples : int or None, default=None
        If not ``None``, limit the training split to at most this many samples.
        Useful for quick debugging runs.
    max_val_samples : int or None, default=None
        If not ``None``, limit the validation split to at most this many samples.
    max_test_samples : int or None, default=None
        If not ``None``, limit the test split to at most this many samples.
    use_local_disk_copy : bool, default=False
        If ``True``, save the downloaded HF dataset to ``local_disk_path`` and
        load from there on subsequent runs (avoids re-downloading and reprocessing).
    local_disk_path : str, default="./data/ai4mars_hf"
        Path to store or load the local disk copy of the raw HF dataset.
    scan_spurious : bool, default=False
        Controls how spurious/corrupted samples are handled:

        - If ``True``, each split is scanned on this run, and a list of valid
          indices is built and saved into ``valid_indices_cache_dir``.
        - If ``False``, we assume the scanning was already done, and valid
          indices are loaded from disk (if present), avoiding the expensive scan.
    valid_indices_cache_dir : str, default="./ai4mars_valid_indices"
        Directory where per-split valid index files (``.npy``) are stored and
        loaded. Files are named like ``valid_indices_train.npy``,
        ``valid_indices_val.npy``, and ``valid_indices_test.npy``.

    Returns
    -------
    DataLoaders
        A dataclass bundle with ``train``, ``val``, and ``test`` PyTorch dataloaders.

    Examples
    --------
    Basic usage:

    .. code-block:: python

        loaders = create_ai4mars_dataloaders(
            batch_size=8,
            image_size=256,
            num_workers=4,
            val_fraction=0.1,
            to_rgb=False,
            scan_spurious=True,  # first run: build valid index cache
        )

        train_loader = loaders.train
        val_loader = loaders.val
        test_loader = loaders.test

    For later runs, you can skip scanning:

    .. code-block:: python

        loaders = create_ai4mars_dataloaders(
            batch_size=8,
            image_size=256,
            num_workers=4,
            val_fraction=0.1,
            to_rgb=False,
            scan_spurious=False,  # reuse cached valid indices
        )
    """

    # --- Load HF dataset (raw) ---
    if use_local_disk_copy and os.path.exists(local_disk_path):
        raw = load_from_disk(local_disk_path)
    else:
        raw = load_dataset(
            "hassanjbara/AI4MARS",
            cache_dir=cache_dir,
        )
        if use_local_disk_copy:
            os.makedirs(os.path.dirname(local_disk_path), exist_ok=True)
            raw.save_to_disk(local_disk_path)

    # --- Split into train / test / val as before ---
    if "train" in raw:
        full_train = raw["train"]
        if "test" in raw:
            test_hf = raw["test"]
        else:
            split = full_train.train_test_split(test_size=0.1, seed=seed)
            full_train, test_hf = split["train"], split["test"]
    else:
        key = list(raw.keys())[0]
        full_train = raw[key]
        split = full_train.train_test_split(test_size=0.2, seed=seed)
        full_train, test_hf = split["train"], split["test"]

    split2 = full_train.train_test_split(test_size=val_fraction, seed=seed + 1)
    train_hf, val_hf = split2["train"], split2["test"]

    # --- Optional subsampling for quick experiments ---
    if max_train_samples is not None:
        train_hf = train_hf.select(range(min(max_train_samples, len(train_hf))))
    if max_val_samples is not None:
        val_hf = val_hf.select(range(min(max_val_samples, len(val_hf))))
    if max_test_samples is not None:
        test_hf = test_hf.select(range(min(max_test_samples, len(test_hf))))

    # --- Wrap in AI4MarsHFDataset with spurious-handling flags ---
    train_ds = AI4MarsHFDataset(
        train_hf,
        image_size=image_size,
        to_rgb=to_rgb,
        scan_spurious=scan_spurious,
        cache_dir=valid_indices_cache_dir,
        split_name="train",
    )
    val_ds = AI4MarsHFDataset(
        val_hf,
        image_size=image_size,
        to_rgb=to_rgb,
        scan_spurious=scan_spurious,
        cache_dir=valid_indices_cache_dir,
        split_name="val",
    )
    test_ds = AI4MarsHFDataset(
        test_hf,
        image_size=image_size,
        to_rgb=to_rgb,
        scan_spurious=scan_spurious,
        cache_dir=valid_indices_cache_dir,
        split_name="test",
    )

    # --- Dataloaders ---
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return DataLoaders(train=train_loader, val=val_loader, test=test_loader)


## File Cell: `optimizers.py`

In [ ]:
# optimizers.py
"""
Optimizer helpers for model training.

This module provides:

1. **create_optimizer**  
   Smart optimizer selection with priority:
   - Muon (if installed and explicitly enabled),
   - NAdam (PyTorch's NAdamW-like implementation),
   - AdamW as a safe fallback.

2. **create_cosine_scheduler_with_warmup**  
   A cosine–annealing learning rate schedule with linear warmup,
   mathematically equivalent to the Hugging Face transformers scheduler.

Both utilities are framework-agnostic and work with any PyTorch model.
"""

from __future__ import annotations

import math
from typing import Optional

import torch

# Try importing Muon (optional dependency)
try:
    from muon import Muon  # KellerJordan/Muon optimizer
    _HAS_MUON = True
except Exception:
    Muon = None  # type: ignore
    _HAS_MUON = False


# ---------------------------------------------------------------------------
# Optimizer Factory
# ---------------------------------------------------------------------------
def create_optimizer(
    model: torch.nn.Module,
    lr: float = 3e-4,
    weight_decay: float = 1e-2,
    use_muon: bool = True,
) -> torch.optim.Optimizer:
    r"""
    Create an optimizer for a given model with prioritized fallback logic.

    The optimizers are tried in the following priority:

    1. **Muon** (if installed and ``use_muon=True``)  
       Muon is a second-order optimizer approximating natural gradient steps.

    2. **NAdam**  
       PyTorch's NAdam implementation (NadamW-style), supporting weight decay.

    3. **AdamW**  
       Stable, widely used, standard fallback.

    Parameters
    ----------
    model : torch.nn.Module
        Model whose trainable parameters will be optimized.
    lr : float, optional
        Learning rate (default: ``3e-4``).
    weight_decay : float, optional
        Weight decay coefficient (default: ``1e-2``).
    use_muon : bool, optional
        Whether the user prefers to use Muon if available.

    Returns
    -------
    torch.optim.Optimizer
        Constructed optimizer instance.

    Notes
    -----
    - Only parameters with ``requires_grad=True`` are passed to the optimizer.
    - If Muon is requested but not installed, AdamW is used and a warning printed.
    """
    params_list = [p for p in model.parameters() if p.requires_grad]

    # --------------------
    # 1) Try Muon
    # --------------------
    if use_muon and _HAS_MUON:
        print("[optimizers] Using Muon optimizer.")
        return Muon(params_list, lr=lr, weight_decay=weight_decay)  # type: ignore

    # --------------------
    # 2) Try NAdam (NadamW-style)
    # --------------------
    if hasattr(torch.optim, "NAdam"):
        print("[optimizers] Using NAdam (NadamW-style) optimizer.")
        return torch.optim.NAdam(params_list, lr=lr, weight_decay=weight_decay)

    # --------------------
    # 3) Fallback: AdamW
    # --------------------
    if use_muon and not _HAS_MUON:
        print(
            "[optimizers] Muon requested but not installed.\n"
            "Install via: pip install git+https://github.com/KellerJordan/Muon"
        )

    print("[optimizers] Using AdamW optimizer.")
    return torch.optim.AdamW(params_list, lr=lr, weight_decay=weight_decay)


# ---------------------------------------------------------------------------
# Cosine Annealing LR Schedule with Warmup
# ---------------------------------------------------------------------------
def create_cosine_scheduler_with_warmup(
    optimizer: torch.optim.Optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
    num_cycles: float = 0.5,
) -> torch.optim.lr_scheduler.LambdaLR:
    r"""
    Create a cosine-annealing LR scheduler with linear warmup.

    This scheduler combines a linear warmup phase with a cosine decay phase.

    **Learning rate schedule**

    Given current step :math:`t`, warmup :math:`W`, and total steps :math:`T`,
    the schedule is:

    **Warmup (linear)**

    .. math::

        \text{lr}(t) = \frac{t}{W}, \quad 0 \le t < W

    **Cosine decay**

    .. math::

        \text{progress} = \frac{t - W}{T - W}

        \text{lr}(t) =
        \tfrac{1}{2}\left(1 + \cos\big( 2\pi \cdot C \cdot \text{progress} \big)\right)

    where:

    - :math:`C` = ``num_cycles`` controls the number of cosine waves
      (``0.5`` = standard: decay → 0 once)

    Parameters
    ----------
    optimizer : torch.optim.Optimizer
        Optimizer whose learning rate will be scheduled.
    num_warmup_steps : int
        Number of linear warmup steps, typically 5–10% of total training steps.
    num_training_steps : int
        Total number of steps (``epochs * steps_per_epoch``).
    num_cycles : float, optional
        Number of cosine cycles.
        Default ``0.5`` = half-cycle (decay to 0 exactly once).

    Returns
    -------
    torch.optim.lr_scheduler.LambdaLR
        Scheduler that updates the LR dynamically during training.

    Notes
    -----
    - This implementation is mathematically similar to
      ``transformers.get_cosine_schedule_with_warmup``.
    - The value returned by the lambda is multiplied with the optimizer's base LR.
    """

    def lr_lambda(current_step: int) -> float:
        # ---- Linear warmup ----
        if current_step < num_warmup_steps:
            return float(current_step) / max(1, num_warmup_steps)

        # ---- Cosine decay ----
        progress = float(current_step - num_warmup_steps) / max(
            1, num_training_steps - num_warmup_steps
        )
        progress = min(max(progress, 0.0), 1.0)

        return 0.5 * (1.0 + math.cos(math.pi * 2.0 * num_cycles * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


## File Cell: `train_utils.py` subset used here (`save_checkpoint`, `load_checkpoint`)

In [ ]:
# Notebook extraction of the checkpoint helpers from train_utils.py
from typing import Any, Dict, Optional

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import _LRScheduler


def save_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[_LRScheduler] = None,
    epoch: Optional[int] = None,
    metrics: Optional[Dict[str, float]] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> None:
    state: Dict[str, Any] = {
        "model_state": model.state_dict(),
    }

    if optimizer is not None:
        state["optimizer_state"] = optimizer.state_dict()
    if scheduler is not None:
        state["scheduler_state"] = scheduler.state_dict()
    if epoch is not None:
        state["epoch"] = epoch
    if metrics is not None:
        state["metrics"] = metrics
    if extra is not None:
        state["extra"] = extra

    torch.save(state, path)
    print(f"[checkpoint] Saved checkpoint to {path}")


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[_LRScheduler] = None,
    map_location: str | torch.device = "cpu",
) -> Dict[str, Any]:
    checkpoint = torch.load(path, map_location=map_location)
    model.load_state_dict(checkpoint["model_state"])
    print(f"[checkpoint] Loaded model weights from {path}")

    if optimizer is not None and "optimizer_state" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        print("[checkpoint] Restored optimizer state.")

    if scheduler is not None and "scheduler_state" in checkpoint:
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        print("[checkpoint] Restored scheduler state.")

    return {
        "epoch": checkpoint.get("epoch", None),
        "metrics": checkpoint.get("metrics", None),
        "extra": checkpoint.get("extra", None),
    }


## File Cell: `exo_models.py`

In [ ]:
from __future__ import annotations
from typing import Dict, Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


class LayerNorm2d(nn.Module):
    """
    LayerNorm over channels for NCHW tensors.
    Input: [B, C, H, W]
    """
    def __init__(self, channels: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=1, keepdim=True)
        var = (x - mean).pow(2).mean(dim=1, keepdim=True)
        x = (x - mean) / torch.sqrt(var + self.eps)
        return self.weight[:, None, None] * x + self.bias[:, None, None]


# ------------------------------------------------------------
# Utility: stochastic depth
# ------------------------------------------------------------
class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.drop_prob == 0.0 or not self.training:
            return x

        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)

        random_tensor = keep_prob + torch.rand(
            shape,
            dtype=x.dtype,
            device=x.device,
        )
        random_tensor.floor_()

        return x.div(keep_prob) * random_tensor


# ------------------------------------------------------------
# Window helpers
# ------------------------------------------------------------
def window_partition(
    x: torch.Tensor,
    window_size: int,
) -> torch.Tensor:
    """
    Partition NHWC feature map into non-overlapping windows.

    Input:
        x: [B, H, W, C]

    Output:
        windows: [B * num_windows, window_size, window_size, C]
    """
    b, h, w, c = x.shape

    x = x.view(
        b,
        h // window_size,
        window_size,
        w // window_size,
        window_size,
        c,
    )

    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    windows = windows.view(-1, window_size, window_size, c)

    return windows


def window_reverse(
    windows: torch.Tensor,
    window_size: int,
    h: int,
    w: int,
) -> torch.Tensor:
    """
    Reverse window partition.

    Input:
        windows: [B * num_windows, window_size, window_size, C]

    Output:
        x: [B, H, W, C]
    """
    num_windows_h = h // window_size
    num_windows_w = w // window_size

    b = int(windows.shape[0] / (num_windows_h * num_windows_w))

    x = windows.view(
        b,
        num_windows_h,
        num_windows_w,
        window_size,
        window_size,
        -1,
    )

    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    x = x.view(b, h, w, -1)

    return x


class ConvNeXtBlock(nn.Module):
    """
    ConvNeXt-style residual block for NCHW feature maps.

    Structure:
        depthwise 7x7 conv
        LayerNorm2d
        pointwise conv expansion
        GELU
        pointwise conv projection
        layer scale
        residual connection
    """
    def __init__(
        self,
        channels: int,
        expansion: int = 4,
        layer_scale_init: float = 1e-6,
        drop_path: float = 0.0,
    ):
        super().__init__()

        self.dwconv = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = LayerNorm2d(channels)

        self.pwconv1 = nn.Conv2d(
            channels,
            expansion * channels,
            kernel_size=1,
        )
        self.act = nn.GELU()
        self.pwconv2 = nn.Conv2d(
            expansion * channels,
            channels,
            kernel_size=1,
        )

        self.gamma = nn.Parameter(
            layer_scale_init * torch.ones(channels),
            requires_grad=True,
        )

        self.drop_path = DropPath(drop_path)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x

        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)

        x = self.gamma[:, None, None] * x

        return residual + self.drop_path(x)
    



class ConvNeXtDownsample(nn.Module):
    """
    Learned downsampling block.

    Reduces spatial size by 2 and changes channel dimension.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        self.block = nn.Sequential(
            LayerNorm2d(in_channels),
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=2,
                padding=1,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ConvNeXtStage(nn.Module):
    """
    Optional downsampling followed by several ConvNeXt blocks.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        depth: int,
        downsample: bool = True,
        drop_path: float = 0.0,
    ):
        super().__init__()

        if downsample:
            self.downsample = ConvNeXtDownsample(in_channels, out_channels)
        else:
            assert in_channels == out_channels
            self.downsample = nn.Identity()

        self.blocks = nn.Sequential(
            *[
                ConvNeXtBlock(
                    out_channels,
                    drop_path=drop_path,
                )
                for _ in range(depth)
            ]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.downsample(x)
        x = self.blocks(x)
        return x
    

class PatchMerging2D(nn.Module):
    """
    Simple CNN-style patch merging.

    In a full Swin implementation, patch merging is often done by concatenating
    neighboring 2x2 tokens and applying a linear layer. This Conv2d version is
    simpler for NCHW feature maps.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()

        self.norm = LayerNorm2d(in_channels)
        self.reduction = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.norm(x)
        x = self.reduction(x)
        return x


class WindowAttention(nn.Module):
    """
    Window-based multi-head self-attention with relative position bias.

    Input:
        x: [B_windows, N, C]
           where N = window_size * window_size
    """

    def __init__(
        self,
        dim: int,
        window_size: int,
        num_heads: int,
        qkv_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
    ):
        super().__init__()

        if dim % num_heads != 0:
            raise ValueError(
                f"dim={dim} must be divisible by num_heads={num_heads}"
            )

        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads

        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # Relative position bias table.
        # Size: (2M-1) * (2M-1), num_heads
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros(
                (2 * window_size - 1) * (2 * window_size - 1),
                num_heads,
            )
        )

        # Pair-wise relative position index for each token inside a window.
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(
            torch.meshgrid(coords_h, coords_w, indexing="ij")
        )  # [2, M, M]

        coords_flatten = torch.flatten(coords, 1)  # [2, M*M]

        relative_coords = (
            coords_flatten[:, :, None] - coords_flatten[:, None, :]
        )  # [2, M*M, M*M]

        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1

        relative_position_index = relative_coords.sum(-1)  # [M*M, M*M]

        self.register_buffer(
            "relative_position_index",
            relative_position_index,
            persistent=False,
        )

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)

        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        x:
            [B_windows, N, C]
        mask:
            Optional attention mask for shifted windows.
            Shape: [num_windows, N, N]
        """
        b_windows, n, c = x.shape

        qkv = self.qkv(x)
        qkv = qkv.reshape(
            b_windows,
            n,
            3,
            self.num_heads,
            c // self.num_heads,
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q * self.scale
        attn = q @ k.transpose(-2, -1)  # [B_windows, heads, N, N]

        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.reshape(-1)
        ]
        relative_position_bias = relative_position_bias.view(
            self.window_size * self.window_size,
            self.window_size * self.window_size,
            -1,
        )
        relative_position_bias = relative_position_bias.permute(
            2, 0, 1
        ).contiguous()  # [heads, N, N]

        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            num_windows = mask.shape[0]

            attn = attn.view(
                b_windows // num_windows,
                num_windows,
                self.num_heads,
                n,
                n,
            )

            attn = attn + mask.unsqueeze(0).unsqueeze(2)

            attn = attn.view(
                -1,
                self.num_heads,
                n,
                n,
            )

        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        x = attn @ v
        x = x.transpose(1, 2).reshape(b_windows, n, c)

        x = self.proj(x)
        x = self.proj_drop(x)

        return x
    
class MLP(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        drop: float = 0.0,
    ):
        super().__init__()

        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)

        return x
    
class SwinTransformerBlock(nn.Module):
    """
    Swin Transformer block for NCHW tensors.

    Alternates between:
      - normal window attention
      - shifted-window attention

    Input/output:
        [B, C, H, W]
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        window_size: int = 8,
        shift_size: int = 0,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        drop_path: float = 0.0,
    ):
        super().__init__()

        if shift_size >= window_size:
            raise ValueError("shift_size must be smaller than window_size.")

        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(dim)

        self.attn = WindowAttention(
            dim=dim,
            window_size=window_size,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )

        self.drop_path = DropPath(drop_path)

        self.norm2 = nn.LayerNorm(dim)

        self.mlp = MLP(
            dim=dim,
            hidden_dim=int(dim * mlp_ratio),
            drop=drop,
        )

    def _make_attention_mask(
        self,
        hp: int,
        wp: int,
        device: torch.device,
        dtype: torch.dtype,
    ) -> torch.Tensor:
        """
        Create attention mask for shifted-window attention.
        Shape: [num_windows, M*M, M*M]
        """
        img_mask = torch.zeros(
            (1, hp, wp, 1),
            device=device,
            dtype=dtype,
        )

        m = self.window_size
        s = self.shift_size

        h_slices = (
            slice(0, -m),
            slice(-m, -s),
            slice(-s, None),
        )
        w_slices = (
            slice(0, -m),
            slice(-m, -s),
            slice(-s, None),
        )

        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1

        mask_windows = window_partition(img_mask, m)
        mask_windows = mask_windows.view(-1, m * m)

        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)

        attn_mask = attn_mask.masked_fill(
            attn_mask != 0,
            float(-100.0),
        )
        attn_mask = attn_mask.masked_fill(
            attn_mask == 0,
            float(0.0),
        )

        return attn_mask

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, C, H, W]
        """
        b, c, h, w = x.shape

        if c != self.dim:
            raise ValueError(
                f"Expected channel dim {self.dim}, got {c}."
            )

        shortcut = x

        # NCHW -> NHWC for LayerNorm and attention.
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm1(x)

        # Pad so H and W are divisible by window_size.
        pad_b = (self.window_size - h % self.window_size) % self.window_size
        pad_r = (self.window_size - w % self.window_size) % self.window_size

        x = F.pad(
            x,
            (0, 0, 0, pad_r, 0, pad_b),
        )

        _, hp, wp, _ = x.shape

        # Shift.
        if self.shift_size > 0:
            shifted_x = torch.roll(
                x,
                shifts=(-self.shift_size, -self.shift_size),
                dims=(1, 2),
            )

            attn_mask = self._make_attention_mask(
                hp=hp,
                wp=wp,
                device=x.device,
                dtype=x.dtype,
            )
        else:
            shifted_x = x
            attn_mask = None

        # Partition windows.
        x_windows = window_partition(
            shifted_x,
            self.window_size,
        )
        x_windows = x_windows.view(
            -1,
            self.window_size * self.window_size,
            c,
        )

        # Window attention.
        attn_windows = self.attn(
            x_windows,
            mask=attn_mask,
        )

        # Reverse windows.
        attn_windows = attn_windows.view(
            -1,
            self.window_size,
            self.window_size,
            c,
        )

        shifted_x = window_reverse(
            attn_windows,
            self.window_size,
            hp,
            wp,
        )

        # Reverse shift.
        if self.shift_size > 0:
            x = torch.roll(
                shifted_x,
                shifts=(self.shift_size, self.shift_size),
                dims=(1, 2),
            )
        else:
            x = shifted_x

        # Remove padding.
        if pad_b > 0 or pad_r > 0:
            x = x[:, :h, :w, :].contiguous()

        # NHWC -> NCHW.
        x = x.permute(0, 3, 1, 2).contiguous()

        # Residual 1.
        x = shortcut + self.drop_path(x)

        # MLP block.
        shortcut = x

        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.norm2(x)
        x = self.mlp(x)
        x = x.permute(0, 3, 1, 2).contiguous()

        # Residual 2.
        x = shortcut + self.drop_path(x)

        return x
    
    
class SwinStage(nn.Module):
    """
    Stack of Swin Transformer blocks.

    Input/output:
        [B, C, H, W]
    """

    def __init__(
        self,
        dim: int,
        depth: int,
        num_heads: int,
        window_size: int = 8,
        mlp_ratio: float = 4.0,
        drop_path: float = 0.0,
    ):
        super().__init__()

        blocks = []

        for i in range(depth):
            shift_size = 0 if i % 2 == 0 else window_size // 2

            blocks.append(
                SwinTransformerBlock(
                    dim=dim,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=shift_size,
                    mlp_ratio=mlp_ratio,
                    drop_path=drop_path,
                )
            )

        self.blocks = nn.Sequential(*blocks)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.blocks(x)
    

class ConvNeXtSwinEncoder(nn.Module):
    """
    CNN/Swin hybrid encoder for HiRISE-style segmentation.

    Output feature pyramid:
      x1: 1/2
      x2: 1/4
      x3: 1/8
      x4: 1/8 after Swin
      x5: 1/16 after Swin
      x6: 1/32 after Swin, optional
    """
    def __init__(
        self,
        in_channels: int,
        base_channels: int = 48,
        use_stage32: bool = True,
        swin_depths: Sequence[int] = (2, 2, 2),
        swin_num_heads: Sequence[int] = (4, 8, 16),
        window_size: int = 8,
        drop_path: float = 0.0,
    ):
        super().__init__()

        if len(swin_depths) != 3:
            raise ValueError("swin_depths must contain 3 stage depths.")
        if len(swin_num_heads) != 3:
            raise ValueError("swin_num_heads must contain 3 stage head counts.")

        c1 = base_channels          # 1/2
        c2 = base_channels * 2      # 1/4
        c3 = base_channels * 4      # 1/8
        c4 = base_channels * 8      # 1/16
        c5 = base_channels * 16     # 1/32

        # Stem: controlled first downsampling to 1/2 resolution.
        self.stem = nn.Sequential(
            nn.Conv2d(
                in_channels,
                c1,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            LayerNorm2d(c1),
            ConvNeXtBlock(c1),
        )

        # ConvNeXt local texture stages.
        self.stage4 = ConvNeXtStage(
            in_channels=c1,
            out_channels=c2,
            depth=2,
            downsample=True,
        )  # 1/4

        self.stage8 = ConvNeXtStage(
            in_channels=c2,
            out_channels=c3,
            depth=2,
            downsample=True,
        )  # 1/8

        # Swin contextual stages.
        self.swin8 = SwinStage(
            dim=c3,
            depth=swin_depths[0],
            num_heads=swin_num_heads[0],
            window_size=window_size,
            drop_path=drop_path,
        )  # still 1/8

        self.merge16 = PatchMerging2D(
            in_channels=c3,
            out_channels=c4,
        )

        self.swin16 = SwinStage(
            dim=c4,
            depth=swin_depths[1],
            num_heads=swin_num_heads[1],
            window_size=window_size,
            drop_path=drop_path,
        )  # 1/16

        self.use_stage32 = use_stage32

        if use_stage32:
            self.merge32 = PatchMerging2D(
                in_channels=c4,
                out_channels=c5,
            )
            self.swin32 = SwinStage(
                dim=c5,
                depth=swin_depths[2],
                num_heads=swin_num_heads[2],
                window_size=window_size,
                drop_path=drop_path,
            )

    def forward(self, x: torch.Tensor):
        x1 = self.stem(x)       # [B, c1, H/2,  W/2]
        x2 = self.stage4(x1)    # [B, c2, H/4,  W/4]
        x3 = self.stage8(x2)    # [B, c3, H/8,  W/8]

        x4 = self.swin8(x3)     # [B, c3, H/8,  W/8]

        x5 = self.merge16(x4)   # [B, c4, H/16, W/16]
        x5 = self.swin16(x5)

        if self.use_stage32:
            x6 = self.merge32(x5)   # [B, c5, H/32, W/32]
            x6 = self.swin32(x6)
            return x1, x2, x3, x4, x5, x6

        return x1, x2, x3, x4, x5


class PatchMasker(nn.Module):
    """
    Random patch masker for image-like tensors.

    Input:
        x: [B, C, H, W]

    Output:
        x_masked: [B, C, H, W]
        mask:     [B, 1, H, W], where 1 means "masked / reconstruct this"
    """
    def __init__(
        self,
        in_channels: int,
        patch_size: int = 16,
        mask_ratio: float = 0.6,
    ):
        super().__init__()

        if not 0.0 < mask_ratio < 1.0:
            raise ValueError("mask_ratio must be between 0 and 1.")

        self.in_channels = in_channels
        self.patch_size = patch_size
        self.mask_ratio = mask_ratio

        # One learnable value per input channel.
        self.mask_token = nn.Parameter(torch.zeros(1, in_channels, 1, 1))

    def make_mask(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape

        if h % self.patch_size != 0 or w % self.patch_size != 0:
            raise ValueError(
                f"H and W must be divisible by patch_size={self.patch_size}. "
                f"Got H={h}, W={w}."
            )

        hp = h // self.patch_size
        wp = w // self.patch_size

        # Patch-level mask: [B, 1, H_patch, W_patch]
        patch_mask = torch.rand(
            b,
            1,
            hp,
            wp,
            device=x.device,
            dtype=x.dtype,
        ) < self.mask_ratio

        patch_mask = patch_mask.to(dtype=x.dtype)

        # Upsample to pixel-level mask: [B, 1, H, W]
        mask = F.interpolate(
            patch_mask,
            size=(h, w),
            mode="nearest",
        )

        return mask

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        mask = self.make_mask(x)

        # Replace masked pixels with learnable mask token.
        x_masked = x * (1.0 - mask) + self.mask_token * mask

        return x_masked, mask
    

class WeakReconstructionDecoder(nn.Module):
    """
    Weak decoder for masked reconstruction pretraining.

    It is intentionally weaker than a segmentation decoder.

    It can use:
        - bottleneck feature only
        - bottleneck + one low-resolution skip, usually 1/8

    It should NOT use high-resolution 1/2 or 1/4 skips during pretraining.
    """
    def __init__(
        self,
        bottleneck_channels: int,
        out_channels: int,
        decoder_channels: int = 256,
        skip8_channels: Optional[int] = None,
        use_skip8: bool = True,
    ):
        super().__init__()

        self.use_skip8 = use_skip8 and skip8_channels is not None

        self.bottleneck_proj = nn.Sequential(
            nn.Conv2d(
                bottleneck_channels,
                decoder_channels,
                kernel_size=1,
                bias=False,
            ),
            LayerNorm2d(decoder_channels),
            nn.GELU(),
            ConvNeXtBlock(decoder_channels),
        )

        if self.use_skip8:
            self.skip8_proj = nn.Sequential(
                nn.Conv2d(
                    skip8_channels,
                    decoder_channels,
                    kernel_size=1,
                    bias=False,
                ),
                LayerNorm2d(decoder_channels),
                nn.GELU(),
            )

            self.fuse = nn.Sequential(
                ConvNeXtBlock(decoder_channels),
                ConvNeXtBlock(decoder_channels),
            )
        else:
            self.skip8_proj = None
            self.fuse = nn.Sequential(
                ConvNeXtBlock(decoder_channels),
            )

        # 1x1 head keeps the full-resolution reconstruction part weak.
        self.reconstruction_head = nn.Conv2d(
            decoder_channels,
            out_channels,
            kernel_size=1,
        )

    def forward(
        self,
        bottleneck: torch.Tensor,
        output_size: Tuple[int, int],
        skip8: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        x = self.bottleneck_proj(bottleneck)

        if self.use_skip8 and skip8 is not None:
            # Bring bottleneck to 1/8 resolution.
            x = F.interpolate(
                x,
                size=skip8.shape[2:],
                mode="bilinear",
                align_corners=False,
            )

            skip = self.skip8_proj(skip8)

            # Additive fusion is weaker than concatenation and avoids an overly
            # powerful reconstruction path.
            x = x + skip
            x = self.fuse(x)

        # Directly upsample to full resolution.
        # This is intentionally simple.
        x = F.interpolate(
            x,
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )

        reconstruction = self.reconstruction_head(x)

        return reconstruction
    


class MaskedReconstructionPretrainer(nn.Module):
    """
    Self-supervised masked reconstruction wrapper.

    This module:
        1. masks input patches
        2. runs the masked image through the encoder
        3. reconstructs the original input
        4. computes loss only on masked pixels

    Intended usage:
        - train encoder + reconstruction decoder
        - discard reconstruction decoder
        - reuse encoder for supervised segmentation
    """
    def __init__(
        self,
        encoder: nn.Module,
        in_channels: int,
        bottleneck_channels: int,
        decoder_channels: int = 256,
        patch_size: int = 16,
        mask_ratio: float = 0.6,
        bottleneck_index: int = -1,
        skip8_index: Optional[int] = None,
        skip8_channels: Optional[int] = None,
        use_skip8: bool = True,
        loss_type: str = "l1",
    ):
        super().__init__()

        self.encoder = encoder

        self.masker = PatchMasker(
            in_channels=in_channels,
            patch_size=patch_size,
            mask_ratio=mask_ratio,
        )

        self.decoder = WeakReconstructionDecoder(
            bottleneck_channels=bottleneck_channels,
            out_channels=in_channels,
            decoder_channels=decoder_channels,
            skip8_channels=skip8_channels,
            use_skip8=use_skip8,
        )

        self.bottleneck_index = bottleneck_index
        self.skip8_index = skip8_index
        self.loss_type = loss_type

        if loss_type not in {"l1", "mse", "smooth_l1"}:
            raise ValueError(
                "loss_type must be one of: 'l1', 'mse', 'smooth_l1'."
            )

    def reconstruction_loss(
        self,
        reconstruction: torch.Tensor,
        target: torch.Tensor,
        mask: torch.Tensor,
    ) -> torch.Tensor:
        """
        Compute reconstruction loss only on masked pixels.

        reconstruction: [B, C, H, W]
        target:         [B, C, H, W]
        mask:           [B, 1, H, W]
        """

        # Broadcast mask from [B,1,H,W] to [B,C,H,W].
        mask = mask.expand_as(target)

        if self.loss_type == "l1":
            loss_map = torch.abs(reconstruction - target)
        elif self.loss_type == "mse":
            loss_map = (reconstruction - target).pow(2)
        elif self.loss_type == "smooth_l1":
            loss_map = F.smooth_l1_loss(
                reconstruction,
                target,
                reduction="none",
            )
        else:
            raise RuntimeError("Invalid loss type.")

        # Avoid division by zero, although mask should never be empty.
        denom = mask.sum().clamp_min(1.0)

        loss = (loss_map * mask).sum() / denom

        return loss

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Forward pass for pretraining.

        Returns a dictionary so you can log reconstruction, mask, etc.
        """
        input_size = x.shape[2:]

        x_masked, mask = self.masker(x)

        features = self.encoder(x_masked)

        if not isinstance(features, (tuple, list)):
            raise TypeError(
                "Encoder must return a tuple/list of feature maps."
            )

        bottleneck = features[self.bottleneck_index]

        skip8 = None
        if self.skip8_index is not None:
            skip8 = features[self.skip8_index]

        reconstruction = self.decoder(
            bottleneck=bottleneck,
            output_size=input_size,
            skip8=skip8,
        )

        loss = self.reconstruction_loss(
            reconstruction=reconstruction,
            target=x,
            mask=mask,
        )

        return {
            "loss": loss,
            "reconstruction": reconstruction,
            "masked_input": x_masked,
            "mask": mask,
        }
    


## Configuration

In [ ]:
# Configuration cell: edit these values before running the training cells
RUN_ARGS = {
    "epochs": 10,
    "batch_size": 8,
    "image_size": 256,
    "num_workers": 0,
    "learning_rate": 3e-4,
    "weight_decay": 1e-2,
    "warmup_fraction": 0.1,
    "base_channels": 48,
    "decoder_channels": 256,
    "patch_size": 16,
    "mask_ratio": 0.6,
    "window_size": 8,
    "swin_depths": (2, 2, 2),
    "swin_num_heads": (4, 8, 16),
    "loss_type": "l1",
    "max_train_samples": None,
    "max_val_samples": None,
    "scan_spurious": False,
    "use_muon": False,
    "use_stage32": True,
    "cache_dir": None,
    "local_disk_path": "data/ai4mars_hf",
    "valid_indices_cache_dir": "ai4mars_valid_indices",
    "checkpoint_path": "checkpoints/best_masked_reconstruction.pt",
    "history_path": "outputs/masked_reconstruction_history.csv",
    "examples_path": "outputs/masked_reconstruction_examples.pt",
    "examples_png_path": "outputs/masked_reconstruction_examples.png",
    "num_examples": 3,
    "seed": 42,
}

for key, value in RUN_ARGS.items():
    print(f"{key}: {value}")


## File Cell: notebook adaptation of `train_masked_reconstruction.py` helpers

In [ ]:
# Notebook adaptation of the helpers from train_masked_reconstruction.py
def resolve_path(path_str: str) -> Path:
    path = Path(path_str).expanduser()
    if not path.is_absolute():
        path = RUN_ROOT / path
    return path


def select_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def run_epoch(model, dataloader, device, optimizer=None, scheduler=None, use_amp=False):
    training = optimizer is not None
    model.train(training)

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    total_loss = 0.0
    total_samples = 0

    context = torch.enable_grad if training else torch.no_grad
    with context():
        for imgs, _ in dataloader:
            imgs = imgs.to(device, non_blocking=True).float()
            batch_size = imgs.size(0)

            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                outputs = model(imgs)
                loss = outputs["loss"]

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * batch_size
            total_samples += batch_size

    return total_loss / max(total_samples, 1)


def save_history(rows: list[dict[str, float]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "val_loss", "lr"])
        writer.writeheader()
        writer.writerows(rows)


def collect_examples(model, dataloader, device, num_examples: int) -> dict:
    model.eval()
    with torch.no_grad():
        for imgs, _ in dataloader:
            imgs = imgs.to(device, non_blocking=True).float()
            outputs = model(imgs)
            count = min(num_examples, imgs.size(0))
            return {
                "original": imgs[:count].detach().cpu(),
                "masked_input": outputs["masked_input"][:count].detach().cpu(),
                "reconstruction": outputs["reconstruction"][:count].detach().cpu(),
                "mask": outputs["mask"][:count].detach().cpu(),
            }

    raise RuntimeError("Unable to collect reconstruction examples from an empty dataloader.")


def save_examples_png(examples: dict, path: Path) -> None:
    original = examples["original"]
    masked_input = examples["masked_input"]
    reconstruction = examples["reconstruction"].clamp(0.0, 1.0)
    mask = examples["mask"]
    num_examples = original.shape[0]

    fig, axes = plt.subplots(
        num_examples,
        4,
        figsize=(12, 3 * num_examples),
        squeeze=False,
    )

    for idx in range(num_examples):
        axes[idx, 0].imshow(original[idx, 0].numpy(), cmap="gray")
        axes[idx, 0].set_title(f"Original {idx + 1}")
        axes[idx, 0].axis("off")

        axes[idx, 1].imshow(masked_input[idx, 0].numpy(), cmap="gray")
        axes[idx, 1].set_title("Masked Input")
        axes[idx, 1].axis("off")

        axes[idx, 2].imshow(reconstruction[idx, 0].numpy(), cmap="gray")
        axes[idx, 2].set_title("Reconstruction")
        axes[idx, 2].axis("off")

        axes[idx, 3].imshow(mask[idx, 0].numpy(), cmap="magma")
        axes[idx, 3].set_title("Mask")
        axes[idx, 3].axis("off")

    best_val_loss = examples.get("best_val_loss")
    if best_val_loss is not None:
        fig.suptitle(
            f"Masked Reconstruction Examples | best val loss = {best_val_loss:.6f}",
            y=1.01,
        )

    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


## Runtime Setup

In [ ]:
# Resolve paths, choose device, and set seeds
seed = RUN_ARGS["seed"]
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = select_device()
use_amp = device.type == "cuda"
print("Using device:", device)

checkpoint_path = resolve_path(RUN_ARGS["checkpoint_path"])
history_path = resolve_path(RUN_ARGS["history_path"])
examples_path = resolve_path(RUN_ARGS["examples_path"])
examples_png_path = resolve_path(RUN_ARGS["examples_png_path"])
local_disk_path = resolve_path(RUN_ARGS["local_disk_path"])
valid_indices_cache_dir = resolve_path(RUN_ARGS["valid_indices_cache_dir"])
cache_dir = resolve_path(RUN_ARGS["cache_dir"]) if RUN_ARGS["cache_dir"] is not None else None


## Load Data

In [ ]:
# Data loading
loaders = create_ai4mars_dataloaders(
    batch_size=RUN_ARGS["batch_size"],
    image_size=RUN_ARGS["image_size"],
    num_workers=RUN_ARGS["num_workers"],
    val_fraction=0.1,
    to_rgb=False,
    seed=RUN_ARGS["seed"],
    cache_dir=str(cache_dir) if cache_dir is not None else None,
    max_train_samples=RUN_ARGS["max_train_samples"],
    max_val_samples=RUN_ARGS["max_val_samples"],
    use_local_disk_copy=True,
    local_disk_path=str(local_disk_path),
    scan_spurious=RUN_ARGS["scan_spurious"],
    valid_indices_cache_dir=str(valid_indices_cache_dir),
)

train_loader = loaders.train
val_loader = loaders.val
test_loader = loaders.test

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")


## Build Model

In [ ]:
# Model, optimizer, and scheduler setup
use_stage32 = RUN_ARGS["use_stage32"]
encoder = ConvNeXtSwinEncoder(
    in_channels=1,
    base_channels=RUN_ARGS["base_channels"],
    use_stage32=use_stage32,
    swin_depths=tuple(RUN_ARGS["swin_depths"]),
    swin_num_heads=tuple(RUN_ARGS["swin_num_heads"]),
    window_size=RUN_ARGS["window_size"],
)

bottleneck_channels = RUN_ARGS["base_channels"] * (16 if use_stage32 else 8)
skip8_channels = RUN_ARGS["base_channels"] * 4

model = MaskedReconstructionPretrainer(
    encoder=encoder,
    in_channels=1,
    bottleneck_channels=bottleneck_channels,
    decoder_channels=RUN_ARGS["decoder_channels"],
    patch_size=RUN_ARGS["patch_size"],
    mask_ratio=RUN_ARGS["mask_ratio"],
    bottleneck_index=-1,
    skip8_index=3,
    skip8_channels=skip8_channels,
    use_skip8=True,
    loss_type=RUN_ARGS["loss_type"],
).to(device)

optimizer = create_optimizer(
    model,
    lr=RUN_ARGS["learning_rate"],
    weight_decay=RUN_ARGS["weight_decay"],
    use_muon=RUN_ARGS["use_muon"],
)

total_steps = RUN_ARGS["epochs"] * len(train_loader)
warmup_steps = int(RUN_ARGS["warmup_fraction"] * total_steps)
scheduler = create_cosine_scheduler_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print("Total training steps:", total_steps)
print("Warmup steps:", warmup_steps)


## Train

In [ ]:
# Training loop
best_val_loss = float("inf")
history_rows: list[dict[str, float]] = []

for epoch in range(1, RUN_ARGS["epochs"] + 1):
    train_loss = run_epoch(
        model=model,
        dataloader=train_loader,
        device=device,
        optimizer=optimizer,
        scheduler=scheduler,
        use_amp=use_amp,
    )
    val_loss = run_epoch(
        model=model,
        dataloader=val_loader,
        device=device,
        optimizer=None,
        use_amp=False,
    )
    current_lr = optimizer.param_groups[0]["lr"]
    history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": current_lr,
        }
    )

    print(
        f"[epoch {epoch:02d}/{RUN_ARGS['epochs']:02d}] "
        f"train_loss={train_loss:.6f} "
        f"val_loss={val_loss:.6f} "
        f"lr={current_lr:.3e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        save_checkpoint(
            path=str(checkpoint_path),
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            metrics={"val_loss": val_loss},
            extra={
                "base_channels": RUN_ARGS["base_channels"],
                "use_stage32": use_stage32,
                "swin_depths": list(RUN_ARGS["swin_depths"]),
                "swin_num_heads": list(RUN_ARGS["swin_num_heads"]),
                "window_size": RUN_ARGS["window_size"],
                "patch_size": RUN_ARGS["patch_size"],
                "mask_ratio": RUN_ARGS["mask_ratio"],
                "loss_type": RUN_ARGS["loss_type"],
            },
        )

save_history(history_rows, history_path)
print(f"Saved training history to {history_path}")


## Save Artifacts

In [ ]:
# Restore best checkpoint and save example artifacts
load_checkpoint(
    path=str(checkpoint_path),
    model=model,
    optimizer=None,
    scheduler=None,
    map_location=device,
)

examples = collect_examples(
    model=model,
    dataloader=val_loader,
    device=device,
    num_examples=RUN_ARGS["num_examples"],
)
examples["best_val_loss"] = best_val_loss
examples["checkpoint_path"] = str(checkpoint_path)

examples_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(examples, examples_path)
print(f"Saved reconstruction examples to {examples_path}")

save_examples_png(examples, examples_png_path)
print(f"Saved reconstruction example grid to {examples_png_path}")


## Inspect Results

In [ ]:
# Inspect the training history and the saved reconstruction PNG inside the notebook
history_df = pd.read_csv(history_path)
display(history_df)

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train loss")
plt.plot(history_df["epoch"], history_df["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction Loss")
plt.title("Masked Reconstruction Pretraining")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

img = plt.imread(examples_png_path)
plt.figure(figsize=(14, 4 + 2 * RUN_ARGS["num_examples"]))
plt.imshow(img)
plt.axis("off")
plt.title("Saved Reconstruction Example Grid")
plt.show()
